---
title: Week 4.2, Real world Application, Simulation of Travertine formation
subtitle: Orchestra Scenario, Lorah & Herman
author:
  - name: Timo Heimovaara
    affiliations: Delft University of Technology, department of Geoscience & Engineering
    orcid: 
    email: t.j.heimovaara@tudelft.nl
license: CC-BY-NC-ND-4.0 (https://creativecommons.org/licenses/by-nc-nd/4.0/).
date: 2026-01-30
kernelspec:
    name: python3
    display_name: 'Python 3.13'
---

## Travertine deposition along a stream
In this assignment you will use Orchestra to model the precipitation of Travertine along a stream. The data we have comes from a paper by [Lorah and Herman, 1988](doi:10.1029/WR024i009p01541) which you can download by clicking on the reference.

In [5]:
# Import libraries required for running all simulations
import os
import sys
from IPython.utils import capture
from IPython.display import display, Markdown
from pathlib import Path
from contextlib import chdir

import numpy as np
import matplotlib.pyplot as plt
import PyORCHESTRA # here, the ORCHESTRA submodule is imported
import pandas as pd
import seaborn as sns

%matplotlib widget
sns.set()

# Prepare a file to capture PyOrchestra output
capture_file = open("pyorchestra_output.log", "w")


# pyOrchestra is implemented in C++
# Save original stdout file descriptor
# original_stdout_fd = sys.stdout.fileno()

# Duplicate original stdout so we can restore it later
# saved_stdout_fd = os.dup(original_stdout_fd)



# We need to import some Orchestra files. We need to know the path layout on 
# the local machine:
def find_book_root(start: Path | None = None) -> Path:
    """
    Walk upward from `start` (or CWD) until a directory containing Jupyter Book
    marker files is found. Returns the path to the book root.
    Raises FileNotFoundError if no root is found.
    """
    config_any = {"_config.yml", "_config.yaml"}      # some projectrs use .yaml
    myst_any = {"myst.yml", "myst.yaml"}            # jupyter-book uses _toc.yml

    cur = Path(Path.cwd()).resolve()

    for parent in [cur, *cur.parents]:
        children = {f.name for f in parent.iterdir()} if parent.exists() else set()
        has_any_config = bool(config_any & children)
        has_any_myst = bool(myst_any & children)
        if has_any_config and has_any_myst:
            return parent

    raise FileNotFoundError(
        f"Could not find Jupyter Book root (no _config.y* and myst.y* found above {cur})"
    )


def path_from_book_root(*parts: str | Path) -> Path:
    root = find_book_root()
    p = (root.joinpath(*parts)).resolve()
    if not p.exists():
        raise FileNotFoundError(f"Path not found: {p}")
    return p

# In order for orchestra to run we need to change directory to the directory with the input file:


# Example usage:
# input_file = path_from_book_root("content", "week 02", "Orchestra_simulation", "chemistry1.inp")
# print("Input file:", input_file)

orchestra_path = path_from_book_root("content", "week 4.2", "Week 4.2_Assignment_Lorah_and_Herman")
# print(orchestra_path)

## Download this script and the required ORCHESTRA files

explain that InVars are the species used to define the totals. 
explain that OutVars is the list of Orchestra variables that are exported to python for post-processing or necessary for simulation.

Once prinicple problem is known we can create and initialize a pyorchestra object.

## Unravelling the geochemistry of Travertine deposition along a stream
The data for this exercise are taken from:
**Reference:** Lorah, M.M. & Herman J.S. (1988) – *The chemical evolution of a tervertine-depositing stream: Geochemical processes and mass transfer reactions*  
<https://doi-org.tudelft.idm.oclc.org/10.1029/WR024i009p01541>  

In the paper by Lorah and Herman (1988) you will see that several sampling trips were made to collect samples along a stream moving away from the hotsprings where the stream originates, past a waterfall and then further down stream. In Table 1, analysis results are presented for to sampling campaigns, one on 14 October 1984 and another on 6  April 1985. We will use these data for our analysis.

The aim of the analysis is to increase our understanding of the formation of travertine deposits along the stream and especially near the water fall.

The analysis you will do follows the approach given in the paper and consists in principle of two main steps:
1. We use that analysis results from the table to assess the the most likely in-situ conditions at the moment of sampling. The analysis results, give totals of the elements expressed as shown in the header of the table. The true speciation will vary, elements will be present in other forms as well, solid minerals may control the composition of the water, but are not sampled with the water. The system may, or may not be in equilibrium. We analyse this problem with Orchestra using an Chemistry input file where we give the total of all measured compounds as master species, and we do not allow any minerals to precipitate, but we do report the SI-values for those minerals.
2. After doing the first analysis, we have quite alot of additional information compared to the chemical analysis results from the lab, we know partial $\text{CO}_2[g]$ pressure and the SI-values for possible minerals for the water samples along the river. It is safe to assume that water down stream from the spring originated from the spring and that its composition should be realated to the water at the spring. With this assumption we can use Orchestra to simulate changes related to the measurements obtained above. For example: if we have water from S-1 and move this to the conditions at D-3 ($\text{CO}_2[g]$ pressure and Calcite-SI), how much calcite would have precipitated in between?

In Orchestra you define the type of calculation you want to do with a *chemistry.inp* file. For this assignment we need two types:
1. To calculate the equilibrium calculations for our system without allowing minerals to precipitate;
2. As similar input file, but where we allow the minerals to precipitate. As we want to compare samples along the line, we also need to be able to control the SI-value which is used to achieve the equilibrium. Normally, the SI-value is assumed to be zero for supersaturated samples, in this assignment we need to allow the model to equilibrate assuming higher, or even lower SI-values. In addition, the $\text{CO}_2[g]$ pressure needs to be fixed as well, which we can achieve by fixing the CO2[g].logact.



```{exercise} **Assumptions**
As stated above, we assume that the composition of the water downstream of the spring is closely related to the composition of the water at the stream. Please carefully evaluate this assumption and try to find reasons why this assumption might not be true? What can you do to mitigate the reasons you found?
```

## Step 1: Assessing the water samples
For this first step I have already prepared an Orchestra input file using the *orchestra2026.jar* Graphical User Interface (GUI). Please have a look at this file with the GUI. 
The approach was quite simple, I have added all the analysed compounds that are found in the table as primary entities in the dissolved phase (*diss*). We define the pH as a fixed logact and we assume that water is the solute phase with a fixed logactivity of 0. We do not select any minerals on the *Phases & Reactions* tab, because we do not allow for any precipitation. We assume that the water samples are stored in bottles with out any gas-phase (head space), we set the gasvolume to be zero on the *Variables* tab. This chemical system is defined in *chemistry1.inp*.

In order to illustrate the steps, we will analyse two water samples: S1 and F1. Please note that the concentrations are given in $\text{mg/l}$ and the temperature in $^\text{o}\text{C}$, we need to traslate these numbers to $\text{mol/l}$ and $\text{K}$ in order to work with Orchestra.

```{hint} Creating the *chemistry.inp* file
Using the Orchestra GUI shown in [](#master_species)  we are able to define a system of the primary entities. You do this by selecting the **Chemistry** tab on the right in the ORCHESTRA-COMPOSER window. The **Chemistry** tab shows a number of tabs which fill the *chemistry1.inp* file. We start by selecting tab **Primary entities/ Master Species**. Remember that we select the Master Species in such a way that we can create all other species for our problem. In addition we need to use these species to define the total moles of each master species present in the system (including the amounts of these master species present in the secondary species).
```


## Initialise the problem 
In order solve this problem with pyOrchestra we first initialize our problem using the *chemistry_Travertine.inp* file. 
This file predefines the aqueous chemical system in such a way that we can use the information from the tables in the paper as inputs to the Orchestra simulation.

Once pyOrchestra is initialized, running a simulation consists of a series of steps where the values of the required set of input variables are passed via *InVARS* to ORCHESTRA, after which a set of corresponding output variables are passed back in *OutVars*.

ORCHESTRA is initialized in pyOrchestra using the inputfile *chemistry_Travertine.inp*, created above with the ORCHESTRA-GUI. After initialization in Python, we know which variables will be passed through *OutVars* and can be used in *InVars*.

The following code implements these steps.


In [6]:
#--- Initialize problem --- 
# for initialization we need to temporarily move to the directory containing the 'chemistry1.inp' file.
# Later we use pO1, InVars1 and OutVars1
with chdir(orchestra_path):

    # Input file is generated with Orchestra GUI
    InputFile = 'chemistry_Travertine.inp'
    NoCells = 1 #only 1 cell to have a 0-D system with 1liter of water
    
    # We define the input variables that will be changed in the script
    # We use the same sequence as used in the paper
    InVars1 = np.array(['T','pH', 'HCO3-.tot', 'Ca+2.tot',  'Mg+2.tot',
                       'Na+.tot', 'K+.tot', 'F-.tot', 'Cl-.tot', 'SO4-2.tot',
                       'watervolume', 'gasvolume'
                       ])
    
    # We select the output from Orchestra we need to use
    OutVars1 = np.array(['T','pH', 
                        'HCO3-.tot', 'HCO3-.con', 'HCO3-.logact', 'HCO3-.diss',
                        'H2CO3.tot', 'H2CO3.con', 'H2CO3.logact', 'H2CO3.diss',
                        'CO3-2.tot', 'CO3-2.con', 'CO3-2.logact', 'CO3-2.diss',
                        'CO2[g].tot', 'CO2[g].con', 'CO2[g].logact', 'CO2[g].diss',
                        'SO4-2.tot', 'SO4-2.con', 'SO4-2.logact','SO4-2.diss',
                        'HSO4-.tot', 'HSO4-.con', 'HSO4-.logact',
                        'Ca+2.tot', 'Ca+2.con', 'Ca+2.logact', 'Ca+2.diss', 
                        'CaF+.con', 'CaOH+.con', 'CaSO4.con',
                        'Calcite[s].si','Calcite[s].tot','Calcite[s].logact',
                        'Fluorite[s].si','Fluorite[s].tot','Fluorite[s].logact',
                        'Gypsum[s].si', 'Gypsum[s].tot','Gypsum[s].logact',
                        'I', 'chargebalance', 'watervolume' ])

    # Associate a variable with the pyOrchestra.ORCHESTRA() class
    pO1 = PyORCHESTRA.ORCHESTRA()

    # Initialize the class with the parameters defined above
    pO1.initialise(InputFile, NoCells, InVars1, OutVars1)


Reading and expanding calculator new stylechemistry_Travertine.inp
Scanning file: chemistry_Travertine.inp
Scanning file: objects2025.txt
Including file: objects2025.txt
Scanning file: chemistry_Travertine.inp
Scanning file: objects2025.txt
Including file: objects2025.txt
Including file: chemistry_Travertine.inp
Scanning file: objects2025.txt
Including file: objects2025.txt
0.062 sec.
	Reading variables .... 0.044 s
testing:
4:T
13:pH
15:HCO3-.tot
8:Ca+2.tot
19:Mg+2.tot
21:Na+.tot
17:K+.tot
12:F-.tot
10:Cl-.tot
23:SO4-2.tot
24:watervolume
25:gasvolume
4:T
13:pH
15:HCO3-.tot
26:HCO3-.con
14:HCO3-.logact
27:HCO3-.diss
28:H2CO3.tot
29:H2CO3.con
30:H2CO3.logact
31:H2CO3.diss
32:CO3-2.tot
33:CO3-2.con
34:CO3-2.logact
35:CO3-2.diss
36:CO2[g].tot
37:CO2[g].con
38:CO2[g].logact
39:CO2[g].diss
23:SO4-2.tot
40:SO4-2.con
22:SO4-2.logact
41:SO4-2.diss
42:HSO4-.tot
43:HSO4-.con
44:HSO4-.logact
8:Ca+2.tot
45:Ca+2.con
7:Ca+2.logact
46:Ca+2.diss
47:CaF+.con
48:CaOH+.con
49:CaSO4.con
50:Calcite[s].si
5

Please note that the output of this code is what ORCHESTRA echos back. ORCHESTRA uses a set of variables in order to store the input variables and the results of the calculations, in this case 33. The top part of the output shows the output requested by us through *OutVars* together with the values used during initialization.

### Run the Problem
We illustrate the approach by analysing two water samples: S1 and F6. Please note that the concentrations are given in mg/l and the temperature in $^o$C, we need to traslate these numbers to mol/l and K in order to work with Orchestra.

### Step 1: import data from paper and create input dataframe for simulations
There are different approaches to import the data. You can type it in to the script directly, you can also copy from the pdf and then edit in the script, or you first copy to a spreadsheet and then work from there. It is a matter of personal taste. The spreadsheet approach, is a good way to move forward as you create a digital backup of the data at the same time.

### Step 2: Translate the data so that we have the temperature in K and the concentrations in mol/l

### Step 3: Copy the values from the imported data to IN1[0] and run Orchestra

## Step 4: Evaluate the output.

In [7]:
# copy data from Lorah and Herman (1988)
# We store the data in a data frame
# Because the data are given in mg/L we need the molar masses to be able to 
# translate to mol/liter

# for conveniece we set mmass for non chemical values to 0.001 because it is then very easy
# to calculate the concentrations in mol per liter...
# Please note: we use Orchestra notation

headers = ['Sample','T_celsius', 'Conductivity', 'pH', 'HCO3-.tot', 'Ca+2.tot',  
           'Mg+2.tot', 'Na+.tot', 'K+.tot', 'F-.tot', 'Cl-.tot', 'SO4-2.tot']
data = [
       ['mmass', 1e-3, 1e-3, 1e-3, 61.02, 40.08, 24.31, 23, 39.1, 19, 34.45, 96.07],
       ['S-1', 24.3, 700, 7.32, 308., 165., 25.4, 5.1, 14.2,  1.0, 3.3, 283.],
       ['F-6', 14.4, 442, 8.25, 210, 122, 21.3, 3.3, 11.4, 0.7, 4, 246]
    ]
df_data_orig = pd.DataFrame(data=data,columns=headers)
df_data_orig.set_index('Sample',inplace=True)

# calculate molar concentrations
df_data = pd.DataFrame()
df_data = df_data_orig.loc[['S-1','F-6']]/df_data_orig.loc['mmass'] * 0.001
# calculate temperature in K
df_data['T'] = df_data['T_celsius'] + 273.15

table_md_data = df_data.to_markdown()
display(Markdown(table_md_data))

# print(df_data)

| Sample   |   T_celsius |   Conductivity |   pH |   HCO3-.tot |   Ca+2.tot |    Mg+2.tot |     Na+.tot |      K+.tot |      F-.tot |    Cl-.tot |   SO4-2.tot |      T |
|:---------|------------:|---------------:|-----:|------------:|-----------:|------------:|------------:|------------:|------------:|-----------:|------------:|-------:|
| S-1      |        24.3 |            700 | 7.32 |  0.00504753 | 0.00411677 | 0.00104484  | 0.000221739 | 0.000363171 | 5.26316e-05 | 9.5791e-05 |  0.00294577 | 297.45 |
| F-6      |        14.4 |            442 | 8.25 |  0.00344149 | 0.00304391 | 0.000876183 | 0.000143478 | 0.00029156  | 3.68421e-05 | 0.00011611 |  0.00256063 | 287.55 |

In [8]:
# %%
# Run the model using chemistry_Travertine.inp for the above samples
IN1 = np.array([np.ones_like(InVars1)]).astype(float)
# prepare output matrix
all_Res = np.zeros([len(df_data),len(OutVars1)])

# set default watervolume and gasvolume
IN1[0][np.where(InVars1 == 'watervolume')] = 1.0 # per liter
IN1[0][np.where(InVars1 == 'gasvolume')] = 0.0 # no gas 

# loop over available samplesCa_Range 
# we use a counter ii in order to store the results
for ii in range(len(df_data)):
    # set InVars
    # please note InVars was initialized assuming the same frequency as
    # Lorah and Herman. 

    # please note, we fill until -2, watervolume and gasvolume have already
    # been defined above. Please note we use "Orchestra" notation
    IN1[0][:-2] = df_data.iloc[ii][['T', 'pH', 'HCO3-.tot', 'Ca+2.tot', 'Mg+2.tot', 
        'Na+.tot', 'K+.tot', 'F-.tot', 'Cl-.tot', 'SO4-2.tot']].values
    
    # run ORCHESTRA
    OUT = pO1.set_and_calculate(IN1)
    all_Res[ii] = OUT[0]

    # Create a dataframe from all_Res
Res_Simulation = pd.DataFrame(all_Res,columns=OutVars1, index=df_data.index)

table_mdini = Res_Simulation[['T','pH', 'HCO3-.con', 
                      'Ca+2.con', 'CO2[g].con','CO2[g].logact', 
                      'Calcite[s].si', 'Gypsum[s].si', 'Fluorite[s].si']].to_markdown()
display(Markdown(table_mdini))


Try a first calculation with iia switched off....
Parsing expressions of chemistry_Travertine.inp..... 
Optimizing expressions of chemistry_Travertine.inp..... 0.237 sec.
6196 variables, 19240 expressions, 8 equations.
First calculation was successful!
Repeat calculation with iia switched on..
Switching on: logI: -2
This was successful!!


| Sample   |      T |   pH |   HCO3-.con |   Ca+2.con |   CO2[g].con |   CO2[g].logact |   Calcite[s].si |   Gypsum[s].si |   Fluorite[s].si |
|:---------|-------:|-----:|------------:|-----------:|-------------:|----------------:|----------------:|---------------:|-----------------:|
| S-1      | 297.45 | 7.32 |  0.00441479 | 0.00332651 |  0.0122069   |        -1.91339 |        0.360526 |      -0.958151 |        -0.924419 |
| F-6      | 287.55 | 8.25 |  0.00323657 | 0.00250576 |  0.000918489 |        -3.03693 |        0.922502 |      -1.06712  |        -1.25799  |

## Interpretation of the results
The results clearly show that the water samples are not in equilibrium with the atmosphere and calcite. The partial pressure of $\text{CO}_2[g]$ is about 400 ppm which would be 0.0004 at which is lower that calculated above. In addition the SI for calcite is above zero.

What is also striking to see is that the SI for calcite increases along the stream, the CO2[g].logact decreases along the stream and we know that Calcite is precipitating from the stream. How much calcite will have precipitated moving from S-1 to F-6?

## Step 2: Calculate amount of calcite precipitated along stream from S-1 to F-6
In order to run this scenario, we need a different type of calculation than used for step 1. The main difference is that we need to allow Orchestra to precipitate calcite. We need to activate the precipitation reactions in the GUI, which allows Orchestra to add the reactions to the equation set. In addition we need allow our Python script to fix the CO2[g].logact value, instead of defining the HCO3-.tot amount.

```{exercise} Explain
Why do we need to fix the CO2[g].logact value instead of defining the HCO3-.tot amount?
```
In addition to allowing Calcite to precipitate, we also want to be able to adjust the SI value to control the equilibrium condition of Calcite. We can achieve this by adding a constant to the precipitation reaction of Calcite in the *chemistry_Travertine.inp* file. In this case we added *deltaSIcalcite* to the file, with a default value of 0. If we set to another value, the model solves an equilibrium for this other value. These changes have been implemented in the *chemistry_Travertine_fixed_CO2_logact.inp* file. We will allow Orchestra to change the pH in order to achieve charge balance.

We can now initialise a second Orchestra calculator for this input file.


In [9]:
#--- Initialize problem --- 
# for initialization we need to temporarily move to the directory containing the 'chemistry2.inp' file.
# Later we use pO2, InVars2 and OutVars2
with chdir(orchestra_path):

    # Input file is generated with Orchestra GUI
    InputFile = 'chemistry_Travertine_fixed_CO2_logact.inp'
    NoCells = 1 #only 1 cell to have a 0-D system with 1liter of water
    
    # We define the input variables that will be changed in the script
    # We use the same sequence as used in the paper
    InVars2 = np.array(['T','CO2[g].logact', 'Ca+2.tot',  'Mg+2.tot',
                       'Na+.tot', 'K+.tot', 'F-.tot', 'Cl-.tot', 'SO4-2.tot',
                       'deltaSIcalcite',
                       'watervolume', 'gasvolume'
                       ])
    
    # We select the output from Orchestra we need to use
    OutVars2 = np.array(['T','pH', 
                        'HCO3-.tot', 'HCO3-.con', 'HCO3-.logact', 'HCO3-.diss',
                        'H2CO3.tot', 'H2CO3.con', 'H2CO3.logact', 'H2CO3.diss',
                        'CO3-2.tot', 'CO3-2.con', 'CO3-2.logact', 'CO3-2.diss',
                        'CO2[g].tot', 'CO2[g].con', 'CO2[g].logact', 'CO2[g].diss',
                        'SO4-2.tot', 'SO4-2.con', 'SO4-2.logact','SO4-2.diss',
                        'HSO4-.tot', 'HSO4-.con', 'HSO4-.logact',
                        'Ca+2.tot', 'Ca+2.con', 'Ca+2.logact', 'Ca+2.diss', 
                        'CaF+.con', 'CaOH+.con', 'CaSO4.con',
                        'Calcite[s].si','Calcite[s].tot','Calcite[s].logact',
                        'Fluorite[s].si','Fluorite[s].tot','Fluorite[s].logact',
                        'Gypsum[s].si', 'Gypsum[s].tot','Gypsum[s].logact',
                        'I', 'chargebalance', 'watervolume' ])

    # Associate a variable with the pyOrchestra.ORCHESTRA() class
    pO2 = PyORCHESTRA.ORCHESTRA()

    # Initialize the class with the parameters defined above
    pO2.initialise(InputFile, NoCells, InVars2, OutVars2)

Reading and expanding calculator new stylechemistry_Travertine_fixed_CO2_logact.inp
Scanning file: chemistry_Travertine_fixed_CO2_logact.inp
Scanning file: objects2025.txt
Including file: objects2025.txt
Scanning file: chemistry_Travertine_fixed_CO2_logact.inp
Scanning file: objects2025.txt
Including file: objects2025.txt
Including file: chemistry_Travertine_fixed_CO2_logact.inp
Scanning file: objects2025.txt
Including file: objects2025.txt
0.087 sec.
	Reading variables .... 0.072 s
testing:
4:T
8:CO2[g].logact
10:Ca+2.tot
19:Mg+2.tot
21:Na+.tot
17:K+.tot
14:F-.tot
12:Cl-.tot
23:SO4-2.tot
25:deltaSIcalcite
26:watervolume
27:gasvolume
4:T
15:pH
28:HCO3-.tot
29:HCO3-.con
30:HCO3-.logact
31:HCO3-.diss
32:H2CO3.tot
33:H2CO3.con
34:H2CO3.logact
35:H2CO3.diss
36:CO3-2.tot
37:CO3-2.con
38:CO3-2.logact
39:CO3-2.diss
40:CO2[g].tot
41:CO2[g].con
8:CO2[g].logact
42:CO2[g].diss
23:SO4-2.tot
43:SO4-2.con
22:SO4-2.logact
44:SO4-2.diss
45:HSO4-.tot
46:HSO4-.con
47:HSO4-.logact
10:Ca+2.tot
48:Ca+2.con

## Strategy to solve the problem
After initializing the Orchestra Calculator, we can solve the problem using our standard approach:
### 1. Define the initial conditions
The initial condition is determined by the measured (total) concentrations in the water sample at S-1.
### 2. Define the boundary conditions
This is the crucial step for the scenario. The sample at F-6 clearly is in a different condition that S-1. The SI value for calcite is different, the CO2-pressure (CO2[g].con, or CO2[g].logact) is different and finally so is the temperature.

We can set these conditions using the InVars2 array, where we use the masses from S-1 and  T, CO2[g].logact and SI Calcite from F-6 all estimated in the previous calculation. The SI-Calcite calculated for F-6 is used for the deltaSIcalcite variable.

We only carry out one simulation.

In [10]:
# Step 1, initial condition.
# InVars need to contain floats
IN2 = np.array([np.ones_like(InVars2)]).astype(float)
# prepare output matrix
all_Res = np.zeros([1,len(OutVars2)])

# set default watervolume and gasvolume
IN2[0][np.where(InVars2 == 'watervolume')] = 1.0 # per liter
IN2[0][np.where(InVars2 == 'gasvolume')] = 0.0 # no gas 

# Set initial concentration values (from point S-1)
inistates = ['Ca+2.tot',  'Mg+2.tot',
             'Na+.tot', 'K+.tot', 'F-.tot', 'Cl-.tot', 'SO4-2.tot']

IN2[0][np.where(np.isin(InVars2,inistates))] = df_data.loc['S-1',inistates].values

# We obtain the target boundary values from Res_Simulation
bndvals = ['T','CO2[g].logact']

IN2[0][np.where(np.isin(InVars2,bndvals))] = Res_Simulation.loc['F-6',bndvals].values
IN2[0][np.where(np.isin(InVars2,'deltaSIcalcite'))] = Res_Simulation.loc['F-6','Calcite[s].si']

# Calculate equilibrium conditions for initial situation (Point A in chapter 4.1)
OUT = pO2.set_and_calculate(IN2)
Res_Sim2 = pd.DataFrame(OUT,columns=OutVars2)

# Need to add the deltaSIcalcite value to the Calcite[s].si value in the output (Orchestra does not do this automatically)
Res_Sim2['Calcite[s].si'] += Res_Simulation.loc['F-6','Calcite[s].si']

Res_Sim2.rename(index={0: 'S-1 to F-6'}, inplace=True)

print("Equilibrate sample from S-1 to F-6")
# print(Res_Sim2[['T','pH','Ca+2.con', 'HCO3-.con', 'CO2[g].con', 'Calcite[s].si', 'Calcite[s].tot', 'CO2[g].logact']])
table_md1 = Res_Sim2[['T','pH','Ca+2.con', 'HCO3-.con', 'CO2[g].con', 'Calcite[s].si', 'Calcite[s].tot', 'CO2[g].logact']].to_markdown()
display(Markdown(table_md1))


print("Initial conditions")

display(Markdown(table_mdini))

Try a first calculation with iia switched off....
Equilibrate sample from S-1 to F-6
Parsing expressions of chemistry_Travertine_fixed_CO2_logact.inp..... 
Optimizing expressions of chemistry_Travertine_fixed_CO2_logact.inp..... 0.25 sec.
6249 variables, 19416 expressions, 8 equations.
First calculation was successful!
Repeat calculation with iia switched on..
Switching on: logI: -2
Switching on: pH: 7
This was successful!!


|            |      T |      pH |   Ca+2.con |   HCO3-.con |   CO2[g].con |   Calcite[s].si |   Calcite[s].tot |   CO2[g].logact |
|:-----------|-------:|--------:|-----------:|------------:|-------------:|----------------:|-----------------:|----------------:|
| S-1 to F-6 | 287.55 | 7.84307 | 0.00191539 |  0.00126824 |  0.000918489 |        0.922502 |       0.00177469 |        -3.03693 |

Initial conditions


| Sample   |      T |   pH |   HCO3-.con |   Ca+2.con |   CO2[g].con |   CO2[g].logact |   Calcite[s].si |   Gypsum[s].si |   Fluorite[s].si |
|:---------|-------:|-----:|------------:|-----------:|-------------:|----------------:|----------------:|---------------:|-----------------:|
| S-1      | 297.45 | 7.32 |  0.00441479 | 0.00332651 |  0.0122069   |        -1.91339 |        0.360526 |      -0.958151 |        -0.924419 |
| F-6      | 287.55 | 8.25 |  0.00323657 | 0.00250576 |  0.000918489 |        -3.03693 |        0.922502 |      -1.06712  |        -1.25799  |

## Analysis
In the simulation above we assume that we can obtain a water sample similar to F-6 from S-1 by changing the SI of calcite and ensuring the temperature and the partial pressure $\text{CO}_2\text[g]$ similar to those of F-6. Clearly this is not completely true which becomes clear when we compare the pH and and the biocarbonate concentrations. 

**Selfstudy questions:** 
+ What could be the reason for these differences? 
+ What was the explanation in the paper?
+ How would you change the above code in order to see the changes for the other species present in this system?

## Assignment
In the above we have analysed two of the samples shown in Lorah and Herman (1988). Your task is to analyse all other results in a similar way. For this assignment you need to do the following:

1. Import all data from Table 1 in Lorah and Herman.
2. Translate all values to K and mg/l where relevant.
3. Add additional information from Lorah and Herman so you can make nice plots. For example, add the distance along the stream which you can estimate from the map shown in Figure 1.
4. Create plots of the raw data similar to figures 2 and 3 to better understand the data.
5. Evaluate all measurements using Orchestra with the *chemistry_Travertine.inp* file.
6. Carefully evaluate the results, make plots for the CO2[g].con, CO2[g].logact, Calcite[s].si etc. so that you understand how the chemistry changes along the stream.
7. Set up a series of Orchestra calculations using the *chemistry_Travertine_fixed_CO2_logact.inp* file in order to estimate the changes in the amounts of HCO3-, Calcite and Ca+2 along the stream.
8. Finally, use the Orchestra to estimate the total amount of Calcite that can precipitate from the samples when they would be brought to equilibrium with the atmosphere in the laboratory at 25 $^\text{o} \text{C}$.


# Analysis of all data from Lorah & Herman

In [23]:
# copy data from Lorah and Herman (1988)
# We store the data in a data frame
# Because the data are given in mg/L we need the molar masses to be able to 
# translate to mol/liter

# for conveniece we set mmass for non chemical values to 0.001 because it is then very easy
# to calculate the concentrations in mol per liter...
# Please note: we use Orchestra notation

# Data October 14, 1984

headers = ['Sample','T_celsius', 'Conductivity', 'pH', 'HCO3-.tot', 'Ca+2.tot',  
           'Mg+2.tot', 'Na+.tot', 'K+.tot', 'F-.tot', 'Cl-.tot', 'SO4-2.tot']
data = [
    ['mmass', 1e-3, 1e-3, 1e-3, 61.02, 40.08, 24.31, 23, 39.1, 19, 34.45, 96.07],
    ['S-1', 24.3, 700, 7.32, 308, 165, 25.4, 5.1, 14.2, 1, 3.3, 283],
    ['S-3', 24.6, 730, 7.23, 316,168,28,3.9,14.8,1.1,3.5,291],
    ['S-2', 24.7, 720, 7.2, 315,169,26.2,3.9,14.3,1.1,5.8,296],
    ['D-1', 24.3, 730, 7.24, 310,168,26,3.4,14.2,1.1,3.2,283],
    ['D-3', 23.6, np.nan, 7.43, 314, 168, 26.2, 2.8, 12.9, 1.2, 3.5, 288],
    ['F-1', 21.4, 695, 7.79, 312,166,25.6,3.5,13.9,1.2,3.4,286],
    ['F-3', 20.5, 695, 8.06, 311,164,25.2,3.7,14,1,3.5,283],
    ['F-4', 19.7, 690, 8.27, 278,160,25.6,3.6,14.3,1,3.8,286],
    ['F-5', 19.6, 690, 8.28, 273,157,25.5,3.1,14,1.1,3.4,283],
    ['F-2', 18.4, 660, 8.37, 264,151,25.6,3.2,14.3,1,3.4,283],
    ['F-6', 14.4, 442, 8.25, 210,122,21.3,3.3,11.4,0.7,4,246]
    ]


df_data_orig_1984 = pd.DataFrame(data=data,columns=headers)
df_data_orig_1984.set_index('Sample',inplace=True)

# calculate molar concentrations
#df_data_1984 = pd.DataFrame()
df_data_1984 = df_data_orig_1984.iloc[1:]/df_data_orig_1984.loc['mmass'] * 0.001
# calculate temperature in K
df_data_1984['T'] = df_data_1984['T_celsius'] + 273.15

table_md_data = df_data_1984.to_markdown()
display(Markdown(table_md_data))

# print(df_data)

| Sample   |   T_celsius |   Conductivity |   pH |   HCO3-.tot |   Ca+2.tot |    Mg+2.tot |     Na+.tot |      K+.tot |      F-.tot |     Cl-.tot |   SO4-2.tot |      T |
|:---------|------------:|---------------:|-----:|------------:|-----------:|------------:|------------:|------------:|------------:|------------:|------------:|-------:|
| S-1      |        24.3 |            700 | 7.32 |  0.00504753 | 0.00411677 | 0.00104484  | 0.000221739 | 0.000363171 | 5.26316e-05 | 9.5791e-05  |  0.00294577 | 297.45 |
| S-3      |        24.6 |            730 | 7.23 |  0.00517863 | 0.00419162 | 0.00115179  | 0.000169565 | 0.000378517 | 5.78947e-05 | 0.000101597 |  0.00302904 | 297.75 |
| S-2      |        24.7 |            720 | 7.2  |  0.00516224 | 0.00421657 | 0.00107775  | 0.000169565 | 0.000365729 | 5.78947e-05 | 0.00016836  |  0.00308109 | 297.85 |
| D-1      |        24.3 |            730 | 7.24 |  0.0050803  | 0.00419162 | 0.00106952  | 0.000147826 | 0.000363171 | 5.78947e-05 | 9.28882e-05 |  0.00294577 | 297.45 |
| D-3      |        23.6 |            nan | 7.43 |  0.00514585 | 0.00419162 | 0.00107775  | 0.000121739 | 0.000329923 | 6.31579e-05 | 0.000101597 |  0.00299781 | 296.75 |
| F-1      |        21.4 |            695 | 7.79 |  0.00511308 | 0.00414172 | 0.00105306  | 0.000152174 | 0.000355499 | 6.31579e-05 | 9.86938e-05 |  0.002977   | 294.55 |
| F-3      |        20.5 |            695 | 8.06 |  0.00509669 | 0.00409182 | 0.00103661  | 0.00016087  | 0.000358056 | 5.26316e-05 | 0.000101597 |  0.00294577 | 293.65 |
| F-4      |        19.7 |            690 | 8.27 |  0.00455588 | 0.00399202 | 0.00105306  | 0.000156522 | 0.000365729 | 5.26316e-05 | 0.000110305 |  0.002977   | 292.85 |
| F-5      |        19.6 |            690 | 8.28 |  0.00447394 | 0.00391717 | 0.00104895  | 0.000134783 | 0.000358056 | 5.78947e-05 | 9.86938e-05 |  0.00294577 | 292.75 |
| F-2      |        18.4 |            660 | 8.37 |  0.00432645 | 0.00376747 | 0.00105306  | 0.00013913  | 0.000365729 | 5.26316e-05 | 9.86938e-05 |  0.00294577 | 291.55 |
| F-6      |        14.4 |            442 | 8.25 |  0.00344149 | 0.00304391 | 0.000876183 | 0.000143478 | 0.00029156  | 3.68421e-05 | 0.00011611  |  0.00256063 | 287.55 |

In [34]:
# copy data from Lorah and Herman (1988)
# We store the data in a data frame
# Because the data are given in mg/L we need the molar masses to be able to 
# translate to mol/liter

# for conveniece we set mmass for non chemical values to 0.001 because it is then very easy
# to calculate the concentrations in mol per liter...
# Please note: we use Orchestra notation

# Data April 6, 1985

headers = ['Sample','T_celsius', 'Conductivity', 'pH', 'HCO3-.tot', 'Ca+2.tot',  
           'Mg+2.tot', 'Na+.tot', 'K+.tot', 'F-.tot', 'Cl-.tot', 'SO4-2.tot']
data = [
    ['mmass', 1e-3, 1e-3, 1e-3, 61.02, 40.08, 24.31, 23, 39.1, 19, 34.45, 96.07],
    ['S-1' ,20,590,6.98,234,102,17,2.5,6.8,0.5,3.3,150,],
    ['S-3',20,600,6.91,239,105,17.1,2.9,7.3,0.6,3.9,159,],
    ['S-2',20,600,6.89,240,106,16.6,2.4,6.9,0.6,3.8,159,],
    ['D-1',19.5,600,7.01,237,103,17.2,3,7.1,0.6,3.6,158,],
    ['D-3',17,595,7.19,237,104,16.6,2.4,6.6,0.6,3.8,158,],
    ['F-1',17,600,7.41,235,104,16.9,2.5,6.5,0.6,3.7,160,],
    ['F-3',16.5,520,7.83,234,102,16.5,2.8,6.9,0.6,3.8,162,],
    ['F-4',13,510,7.98,224,100,16.3,2.7,6.6,0.6,3.7,162,],
    ['F-5',13.5,488,7.92,220,99,16.2,2.5,6.7,0.6,3.8,162,],
    ['F-2',13.5,461,7.98,209,97,16.7,2.3,6.4,0.6,3.9,164,],
    ['F-6',11,393,8.08,183,83,15,2.5,5.8,0.6,4.3,142,],
]
    

df_data_orig_1985 = pd.DataFrame(data=data,columns=headers)
df_data_orig_1985.set_index('Sample',inplace=True)

# calculate molar concentrations
#df_data_1985 = pd.DataFrame()
df_data_1985 = df_data_orig_1985.iloc[1:]/df_data_orig_1985.loc['mmass'] * 0.001
# calculate temperature in K
df_data_1985['T'] = df_data_1985['T_celsius'] + 273.15

table_md_data = df_data_1985.to_markdown()
display(Markdown(table_md_data))

# print(df_data)

| Sample   |   T_celsius |   Conductivity |   pH |   HCO3-.tot |   Ca+2.tot |    Mg+2.tot |     Na+.tot |      K+.tot |      F-.tot |     Cl-.tot |   SO4-2.tot |      T |
|:---------|------------:|---------------:|-----:|------------:|-----------:|------------:|------------:|------------:|------------:|------------:|------------:|-------:|
| S-1      |        20   |            590 | 6.98 |  0.00383481 | 0.00254491 | 0.000699301 | 0.000108696 | 0.000173913 | 2.63158e-05 | 9.5791e-05  |  0.00156136 | 293.15 |
| S-3      |        20   |            600 | 6.91 |  0.00391675 | 0.00261976 | 0.000703414 | 0.000126087 | 0.000186701 | 3.15789e-05 | 0.000113208 |  0.00165504 | 293.15 |
| S-2      |        20   |            600 | 6.89 |  0.00393314 | 0.00264471 | 0.000682847 | 0.000104348 | 0.000176471 | 3.15789e-05 | 0.000110305 |  0.00165504 | 293.15 |
| D-1      |        19.5 |            600 | 7.01 |  0.00388397 | 0.00256986 | 0.000707528 | 0.000130435 | 0.000181586 | 3.15789e-05 | 0.000104499 |  0.00164463 | 292.65 |
| D-3      |        17   |            595 | 7.19 |  0.00388397 | 0.00259481 | 0.000682847 | 0.000104348 | 0.000168798 | 3.15789e-05 | 0.000110305 |  0.00164463 | 290.15 |
| F-1      |        17   |            600 | 7.41 |  0.0038512  | 0.00259481 | 0.000695187 | 0.000108696 | 0.00016624  | 3.15789e-05 | 0.000107402 |  0.00166545 | 290.15 |
| F-3      |        16.5 |            520 | 7.83 |  0.00383481 | 0.00254491 | 0.000678733 | 0.000121739 | 0.000176471 | 3.15789e-05 | 0.000110305 |  0.00168627 | 289.65 |
| F-4      |        13   |            510 | 7.98 |  0.00367093 | 0.00249501 | 0.000670506 | 0.000117391 | 0.000168798 | 3.15789e-05 | 0.000107402 |  0.00168627 | 286.15 |
| F-5      |        13.5 |            488 | 7.92 |  0.00360538 | 0.00247006 | 0.000666392 | 0.000108696 | 0.000171355 | 3.15789e-05 | 0.000110305 |  0.00168627 | 286.65 |
| F-2      |        13.5 |            461 | 7.98 |  0.00342511 | 0.00242016 | 0.00068696  | 0.0001      | 0.000163683 | 3.15789e-05 | 0.000113208 |  0.00170709 | 286.65 |
| F-6      |        11   |            393 | 8.08 |  0.00299902 | 0.00207086 | 0.00061703  | 0.000108696 | 0.000148338 | 3.15789e-05 | 0.000124819 |  0.00147809 | 284.15 |

In [28]:
# %%
# All 1984 data
# Run the model using chemistry_Travertine.inp for the above samples
IN1 = np.array([np.ones_like(InVars1)]).astype(float)
# prepare output matrix
all_Res = np.zeros([len(df_data_1984),len(OutVars1)])

# set default watervolume and gasvolume
IN1[0][np.where(InVars1 == 'watervolume')] = 1.0 # per liter
IN1[0][np.where(InVars1 == 'gasvolume')] = 0.0 # no gas 

# loop over available samplesCa_Range 
# we use a counter ii in order to store the results
for ii in range(len(df_data_1984)):
    # set InVars
    # please note InVars was initialized assuming the same frequency as
    # Lorah and Herman. 

    # please note, we fill until -2, watervolume and gasvolume have already
    # been defined above. Please note we use "Orchestra" notation
    IN1[0][:-2] = df_data_1984.iloc[ii][['T', 'pH', 'HCO3-.tot', 'Ca+2.tot', 'Mg+2.tot', 
        'Na+.tot', 'K+.tot', 'F-.tot', 'Cl-.tot', 'SO4-2.tot']].values
    
    # run ORCHESTRA
    OUT = pO1.set_and_calculate(IN1)
    all_Res[ii] = OUT[0]

    # Create a dataframe from all_Res
Res_Sample_Analysis_1984 = pd.DataFrame(all_Res,columns=OutVars1, index=df_data_1984.index)

table_mdini = Res_Sample_Analysis_1984[['T','pH', 'HCO3-.con', 
                      'Ca+2.con', 'CO2[g].con','CO2[g].logact', 
                      'Calcite[s].si', 'Gypsum[s].si', 'Fluorite[s].si']].to_markdown()
display(Markdown(table_mdini))


| Sample   |      T |   pH |   HCO3-.con |   Ca+2.con |   CO2[g].con |   CO2[g].logact |   Calcite[s].si |   Gypsum[s].si |   Fluorite[s].si |
|:---------|-------:|-----:|------------:|-----------:|-------------:|----------------:|----------------:|---------------:|-----------------:|
| S-1      | 297.45 | 7.32 |  0.00441479 | 0.00332651 |  0.0122069   |        -1.91339 |        0.360526 |      -0.958151 |        -0.924419 |
| S-3      | 297.75 | 7.23 |  0.00444378 | 0.00338015 |  0.0151621   |        -1.81924 |        0.281401 |      -0.946431 |        -0.844706 |
| S-2      | 297.85 | 7.2  |  0.00440003 | 0.00339117 |  0.0161101   |        -1.7929  |        0.249924 |      -0.936585 |        -0.840378 |
| D-1      | 297.45 | 7.24 |  0.00436881 | 0.00339383 |  0.0145186   |        -1.83808 |        0.283965 |      -0.952538 |        -0.835234 |
| D-3      | 296.75 | 7.43 |  0.00458044 | 0.00338007 |  0.00972606  |        -2.01206 |        0.482523 |      -0.946226 |        -0.758304 |
| F-1      | 294.55 | 7.79 |  0.00471021 | 0.00333767 |  0.00423076  |        -2.37358 |        0.820568 |      -0.948263 |        -0.7481   |
| F-3      | 293.65 | 8.06 |  0.00472729 | 0.00328333 |  0.00225168  |        -2.64749 |        1.07403  |      -0.955145 |        -0.905705 |
| F-4      | 292.85 | 8.27 |  0.00421131 | 0.00319154 |  0.00122379  |        -2.91229 |        1.21326  |      -0.956032 |        -0.910343 |
| F-5      | 292.75 | 8.28 |  0.00413732 | 0.00313451 |  0.001174    |        -2.93033 |        1.20795  |      -0.964679 |        -0.832584 |
| F-2      | 291.55 | 8.37 |  0.00399618 | 0.00301021 |  0.000906402 |        -3.04268 |        1.25117  |      -0.973968 |        -0.922522 |
| F-6      | 287.55 | 8.25 |  0.00323657 | 0.00250576 |  0.000918489 |        -3.03693 |        0.922502 |      -1.06712  |        -1.25799  |

In [35]:
# %%
# All 1985 data
# Run the model using chemistry_Travertine.inp for the above samples
IN1 = np.array([np.ones_like(InVars1)]).astype(float)
# prepare output matrix
all_Res = np.zeros([len(df_data_1985),len(OutVars1)])

# set default watervolume and gasvolume
IN1[0][np.where(InVars1 == 'watervolume')] = 1.0 # per liter
IN1[0][np.where(InVars1 == 'gasvolume')] = 0.0 # no gas 

# loop over available samplesCa_Range 
# we use a counter ii in order to store the results
for ii in range(len(df_data_1985)):
    # set InVars
    # please note InVars was initialized assuming the same frequency as
    # Lorah and Herman. 

    # please note, we fill until -2, watervolume and gasvolume have already
    # been defined above. Please note we use "Orchestra" notation
    IN1[0][:-2] = df_data_1985.iloc[ii][['T', 'pH', 'HCO3-.tot', 'Ca+2.tot', 'Mg+2.tot', 
        'Na+.tot', 'K+.tot', 'F-.tot', 'Cl-.tot', 'SO4-2.tot']].values
    
    # run ORCHESTRA
    OUT = pO1.set_and_calculate(IN1)
    all_Res[ii] = OUT[0]

    # Create a dataframe from all_Res
Res_Sample_Analysis_1985 = pd.DataFrame(all_Res,columns=OutVars1, index=df_data_1985.index)

table_mdini = Res_Sample_Analysis_1985[['T','pH', 'HCO3-.con', 
                      'Ca+2.con', 'CO2[g].con','CO2[g].logact', 
                      'Calcite[s].si', 'Gypsum[s].si', 'Fluorite[s].si']].to_markdown()
display(Markdown(table_mdini))


| Sample   |      T |   pH |   HCO3-.con |   Ca+2.con |   CO2[g].con |   CO2[g].logact |   Calcite[s].si |   Gypsum[s].si |   Fluorite[s].si |
|:---------|-------:|-----:|------------:|-----------:|-------------:|----------------:|----------------:|---------------:|-----------------:|
| S-1      | 293.15 | 6.98 |  0.00305152 | 0.00220638 |   0.0177093  |        -1.7518  |       -0.328298 |       -1.30535 |         -1.60612 |
| S-3      | 293.15 | 6.91 |  0.00302122 | 0.00226136 |   0.0205771  |        -1.68662 |       -0.394478 |       -1.27449 |         -1.4403  |
| S-2      | 293.15 | 6.89 |  0.00300453 | 0.00228329 |   0.0214297  |        -1.66898 |       -0.412501 |       -1.27026 |         -1.435   |
| D-1      | 292.65 | 7.01 |  0.00312595 | 0.00221784 |   0.0167911  |        -1.77492 |       -0.294455 |       -1.2829  |         -1.4452  |
| D-3      | 290.15 | 7.19 |  0.00330886 | 0.00224428 |   0.0113124  |        -1.94644 |       -0.1194   |       -1.27433 |         -1.42458 |
| F-1      | 290.15 | 7.41 |  0.00344703 | 0.00223729 |   0.00709771 |        -2.14888 |        0.115937 |       -1.27164 |         -1.4276  |
| F-3      | 289.65 | 7.83 |  0.00359208 | 0.00218041 |   0.00279107 |        -2.55423 |        0.535773 |       -1.27426 |         -1.43451 |
| F-4      | 286.15 | 7.98 |  0.00346717 | 0.00214762 |   0.00180967 |        -2.7424  |        0.616351 |       -1.27137 |         -1.41747 |
| F-5      | 286.65 | 7.92 |  0.00339788 | 0.00212661 |   0.00205266 |        -2.68768 |        0.55118  |       -1.27473 |         -1.42364 |
| F-2      | 286.65 | 7.98 |  0.00323577 | 0.00208089 |   0.00170325 |        -2.76872 |        0.581507 |       -1.27652 |         -1.4326  |
| F-6      | 284.15 | 8.08 |  0.00285201 | 0.00180682 |   0.00115323 |        -2.93808 |        0.541512 |       -1.36996 |         -1.46099 |

In [33]:
df_data_1985.iloc[ii]

T_celsius        17.000000
Conductivity    600.000000
pH                7.410000
HCO3-.tot         0.003851
Ca+2.tot          0.002595
Mg+2.tot          0.000103
Na+.tot           0.000283
K+.tot            0.000015
F-.tot            0.000195
Cl-.tot           0.004644
SO4-2.tot              NaN
T               290.150000
Name: F-1, dtype: float64